# Week 4: BERT Fine-Tuning for Fake News Detection

This notebook fine-tunes `distilbert-base-uncased` on the TruthLens fake news dataset. It demonstrates dataset preparation, tokenizer integration, model training, evaluation, and artifact saving.

## 1. Setup and imports

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.metrics import precision_score, recall_score
from sklearn.model_selection import train_test_split
from transformers import (DistilBertForSequenceClassification, DistilBertTokenizerFast, TrainingArguments, Trainer, DataCollatorWithPadding, EarlyStoppingCallback)

PROJECT_ROOT = None
for candidate in [Path.cwd()] + list(Path.cwd().parents):
    if (candidate / 'backend').exists() and (candidate / 'data').exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise RuntimeError('Could not locate project root containing backend/ and data/')

sys.path.insert(0, str(PROJECT_ROOT))

from backend.preprocess import clean_text, download_nltk_resources

DATA_DIR = PROJECT_ROOT / 'data'
MODEL_DIR = PROJECT_ROOT / 'backend' / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Torch device:', torch.device('cuda' if torch.cuda.is_available() else 'cpu'))

## 2. Download NLP resources and load data

In [ ]:
download_nltk_resources()

real_df = pd.read_csv(DATA_DIR / 'True.csv')
fake_df = pd.read_csv(DATA_DIR / 'Fake.csv')
real_df['label'] = 0
fake_df['label'] = 1
df = pd.concat([real_df, fake_df], ignore_index=True)
df = df[['title', 'text', 'label']].fillna('')

print('Total rows:', len(df))
print(df['label'].value_counts())

## 3. Preprocess text and tokenize

In [ ]:
df['combined_text'] = (df['title'] + ' ' + df['text']).astype(str)
df['cleaned_text'] = df['combined_text'].apply(clean_text)

train_df, test_df = train_test_split(df, test_size=0.20, stratify=df['label'], random_state=42)
train_df, val_df = train_test_split(train_df, test_size=0.125, stratify=train_df['label'], random_state=42)

print('Train size:', len(train_df))
print('Validation size:', len(val_df))
print('Test size:', len(test_df))

In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

def tokenize_batch(batch):
    return tokenizer(batch['cleaned_text'], truncation=True, padding=False, max_length=256)

train_dataset = Dataset.from_pandas(train_df[['cleaned_text', 'label']])
val_dataset = Dataset.from_pandas(val_df[['cleaned_text', 'label']])
test_dataset = Dataset.from_pandas(test_df[['cleaned_text', 'label']])

train_dataset = train_dataset.map(tokenize_batch, batched=True)
val_dataset = val_dataset.map(tokenize_batch, batched=True)
test_dataset = test_dataset.map(tokenize_batch, batched=True)

train_dataset = train_dataset.remove_columns(['cleaned_text'])
val_dataset = val_dataset.remove_columns(['cleaned_text'])
test_dataset = test_dataset.remove_columns(['cleaned_text'])

train_dataset.set_format(type='torch')
val_dataset.set_format(type='torch')
test_dataset.set_format(type='torch')

print(train_dataset.features)

## 4. Create the DistilBERT model

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)
model.to(torch.device('cuda' if torch.cuda.is_available() else 'cpu'))

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir=str(MODEL_DIR / 'distilbert-finetuned'),
    evaluation_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    logging_dir=str(PROJECT_ROOT / 'logs' / 'bert_training'),
    logging_steps=100,
    report_to='none',
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        'accuracy': (predictions == labels).astype(float).mean(),
        'precision': precision_score(labels, predictions),
        'recall': recall_score(labels, predictions),
        'f1': f1_score(labels, predictions),
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

## 5. Train the model

In [ ]:
trainer.train()

## 6. Evaluate on the holdout test set

In [ ]:
metrics = trainer.evaluate(test_dataset)
print(metrics)

model.save_pretrained(MODEL_DIR / 'distilbert-finetuned')
tokenizer.save_pretrained(MODEL_DIR / 'distilbert-finetuned')
print('Saved fine-tuned DistilBERT model to', MODEL_DIR / 'distilbert-finetuned')

## 7. Notes and next steps

- Use the saved model and tokenizer in the FastAPI backend.
- The next milestone is building `backend/main.py` for inference endpoints.
- Optionally, add a small React UI for article input and prediction display.